In [ ]:
FILES={'expand_chunks.py': '"""Rebuild the chunk lists of the hybrid submission with neighbouring chunks merged into longer spans.\n\nA reference passage counts as found when a submitted text of the same document covers >= 40% of it (token\nLCS). Chunk recall (0.18) is far below document recall (0.47): the right documents are found, the passages\nare not. Here the top-ranked chunks bring their neighbours (order - 1, order + 1) along, and contiguous\nchunks of one text are merged into one span (the 50-word overlaps are dropped once). The byte budget per\nquery stays that of the hybrid submission, so lower-ranked chunks make room for the neighbours.\n\nThe document lists are kept as submitted; documents of new spans are appended (as in assemble()).\nOptional per variant: n_docs extends the document list with the next pool documents (best chunk first);\nquota reserves that share of the chunk bytes for ZH/EN texts; doc_quota swaps the lowest VI documents of the\nlist for the best ZH/EN pool documents (both lost on the leaderboard, 29/10); doc_rr re-orders the\ncandidate documents by weighted RRF of their submitted rank and document-reranker scores (doc_rank.parquet,\nwritten by the doc-rerank notebook, kaggle_doc_rerank.ipynb), so the reranker decides which make the cut.\nconf_gamma moves chunk bytes between queries, total unchanged: a query\'s confidence is its number of pool\nchunks with rerank score >= conf_t; confident queries lose budget (their lowest-ranked chunks go) and the\nbudget goes to queries with few strong chunks, which reach deeper into their pool.\nRuns on a Kaggle CPU session (Internet OFF is fine). Inputs:\n  medxir-chunk-corpus      chunks/*/part-*.parquet, hash_groups.parquet\n  medxir-hybrid-retrieval  pool.parquet, medxir-hybrid-submission.zip   (notebook output or uploaded dataset)\n  doc_rank.parquet         only for doc_rr variants (in OUT_DIR or an input)\nOutput: one ZIP per variant in /kaggle/working (or OUT_DIR), plus expand_manifest.json.\n"""\nimport glob\nimport json\nimport math\nimport os\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow as pa\nimport pyarrow.compute as pc\nimport pyarrow.parquet as pq\n\n# nb: number of top chunks whose neighbours are added (0 = only merge, no new neighbours);\n#     \'whole\' = probe: full texts of the top documents instead of ranked chunks.\n# n_docs: document list length (0 = as submitted). quota / doc_quota: min ZH/EN share of chunk bytes / documents.\n# doc_rr: {score column of doc_rank.parquet: RRF weight}; w_base: RRF weight of the submitted document rank;\n# keep_top: that many top candidates kept as submitted, the rest of the list re-ordered.\n# conf_gamma: budget weight (confidence / median confidence) ** -gamma, clipped to CONF_CLIP, renormalised.\n# min_chars: spans shorter than this are not submitted. A span matches when it covers 40% of a reference\n# passage, so a 5-character ZH entry (\'食管炎\') can only match an equally short passage; such spans were 11% of\n# the entries of the 0.2464 ZIP (< 200 chars) for 0.35% of its text.\n# min_score: ranked chunks with a rerank score below it are not taken (nor their neighbours); the bytes a query\n# leaves unused go to the queries that used their whole budget, by the same confidence weights. bge scores are\n# high: at 0.2 the cut removes ~46 chunks in 16% of the queries (the unsure ones) and none elsewhere.\n# zip: \'bzip2\' writes the ZIP with bzip2 instead of deflate: the 0.2477 JSON is 86.5 MB deflated, 56.5 MB with\n# bzip2 (LZMA 54.4 MB, 10x slower), so ~1.5x more text fits under the 100 MB limit. budget_scale replaces\n# BUDGET_SCALE; the size loop shrinks it until the ZIP fits. Under the limit chunk recall grows with the bytes\n# (each gold document holds ~1 reference passage, so more bytes = more documents reached).\n# cover (first, last): after the nb top chunks (with neighbours), the best pool chunk of every listed document\n# at ranks first..last that has none yet, then the rest of the pool. 30/10 (Qwen-8B, 0.2509): ranks 100-200 of\n# the document list got chunk text for 10% of the documents, ranks 50-100 for 48%, while ~15% of them are\n# relevant and a relevant document holds ~1 reference passage.\n# win: every entry is a window of the chunk +- win neighbours of the same text (the entry shape of the 0.2755\n# ensemble ZIP: no tiny spans, no 5+ chunk merges); a centre inside a chosen window is skipped. Centres in the\n# order of w_bge * bge + (1 - w_bge) * rrf, both min-max per query (0.2755 used bge 0.25 over its fused list).\n# cover_top: first the best window of each of the top cover_top listed documents, then one window per text in\n# score order, then second windows. A relevant document holds ~1 reference passage, so a second window of one\n# text rarely adds recall, while ~23% of the top-20 documents of the 0.2755 ZIP had no window.\n# doc_conf (t, lo, hi): per query, the documents with doc-reranker score >= t, at least lo and at most hi. A\n# document pays off in F2 when P(relevant) > ~F2 / 5 (~0.065 at Doc F2 0.32); bge scores scaled to the 0.161 Doc P\n# of the 0.2539 ZIP (mean bge of its top 200 = 0.514) put that at t = 0.208 (~189 documents on average).\n# bm25w (needs bm25_words.py outputs attached): doc_rr column \'bm25w\' = the word-BM25 rank of the best\n# BM25W_TEXTS texts (their documents), fused by RRF with bge; w_bm25w adds the word-BM25 chunk score (best\n# BM25W_CHUNKS chunks, also those outside the pool) to the window order. 30/10: 48% of the word-BM25 top 40\n# documents are in none of our 206-document lists.\nVARIANTS = {\n    \'nb0\': dict(nb=0), \'nb10\': dict(nb=10), \'nb30\': dict(nb=30), \'nb60\': dict(nb=60), \'wholedoc\': dict(nb=\'whole\'),\n    \'nb60-d300\': dict(nb=60, n_docs=300),\n    \'nb60-q20\': dict(nb=60, quota=0.2),\n    \'nb60-q20-dq20\': dict(nb=60, quota=0.2, doc_quota=0.2),\n    \'nb60-drr-bge\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=1.0),\n    \'nb60-drr-bge-only\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0),\n    # Qwen3-Reranker-4B scores only the zone around the cut (ranks 120-360): the top 120 stay as submitted.\n    \'nb60-drr-qwen4b\': dict(nb=60, doc_rr={\'Qwen3-Reranker-4B\': 1.0}, w_base=1.0, keep_top=120),\n    \'nb60-drr-qwen4b-only\': dict(nb=60, doc_rr={\'Qwen3-Reranker-4B\': 1.0}, w_base=0.0, keep_top=120),\n    \'nb60-drr-both\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0, \'Qwen3-Reranker-4B\': 1.0}, w_base=1.0,\n                          keep_top=120),\n    # Bge-only document list, chunk bytes moved between queries. 30/10 on v2 (bge-only 0.2464): glow (to unsure\n    # queries) 0.2352, ghigh (to confident queries) 0.2477; macro F2 rewards the queries that already score.\n    \'nb60-drr-bge-only-glow\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0, conf_gamma=0.5, conf_t=0.5),\n    \'nb60-drr-bge-only-ghigh\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0, conf_gamma=-0.5, conf_t=0.5),\n    \'nb60-drr-bge-only-ghigh1\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0, conf_gamma=-1.0,\n                                     conf_t=0.5),\n    # Precision: no tiny spans; \'nball\' gives every ranked chunk its neighbours (no single chunks filling the\n    # budget), the entry shape of the x1-c70 submission (Chunk P 0.208 vs 0.155).\n    \'nb60-drr-bge-only-ghigh1-min200\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0, conf_gamma=-1.0,\n                                            conf_t=0.5, min_chars=200),\n    \'nb60-drr-bge-only-ghigh-min200-s20\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0,\n                                               conf_gamma=-0.5, conf_t=0.5, min_chars=200, min_score=0.2),\n    \'nb60-drr-bge-only-ghigh1-min200-s20\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0,\n                                                conf_gamma=-1.0, conf_t=0.5, min_chars=200, min_score=0.2),\n    \'nb60-drr-bge-only-ghigh-bz\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0, conf_gamma=-0.5,\n                                       conf_t=0.5, zip=\'bzip2\', budget_scale=1.6),\n    \'nb60-drr-bge-only-ghigh-cover\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0, conf_gamma=-0.5,\n                                          conf_t=0.5, cover=(20, 200)),\n    \'nb30-drr-bge-only-ghigh-cover\': dict(nb=30, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0, conf_gamma=-0.5,\n                                          conf_t=0.5, cover=(20, 200)),\n    \'nb60-drr-bge-only-ghigh-min200\': dict(nb=60, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0, conf_gamma=-0.5,\n                                           conf_t=0.5, min_chars=200),\n    \'nball-drr-bge-only-ghigh-min200\': dict(nb=10 ** 6, doc_rr={\'bge-reranker-v2-m3\': 1.0}, w_base=0.0,\n                                            conf_gamma=-0.5, conf_t=0.5, min_chars=200),\n    \'win1-cov50-drr-bge-only\': dict(nb=0, win=1, w_bge=0.25, cover_top=50, doc_rr={\'bge-reranker-v2-m3\': 1.0},\n                                    w_base=0.0, budget_scale=1.15),\n    \'win1-cov50-drr-bge-conf\': dict(nb=0, win=1, w_bge=0.25, cover_top=50, doc_rr={\'bge-reranker-v2-m3\': 1.0},\n                                    w_base=0.0, budget_scale=1.15, doc_conf=(0.208, 80, 300)),\n    \'win1-cov50-bm25w-drr\': dict(nb=0, win=1, w_bge=0.25, w_bm25w=0.25, cover_top=50,\n                                 doc_rr={\'bge-reranker-v2-m3\': 1.0, \'bm25w\': 1.0}, w_base=0.0, budget_scale=1.15),\n}\nONLY = [\'nb60-d300\']   # variants to build this run; [] = all\nBUDGET_SCALE = 1.08   # 87.5 MB hybrid ZIP * 1.08 stays below the 100 MB limit\nMAX_ZIP_MB = 99.0\nCONF_CLIP = (0.5, 2.0)\nBM25W_CHUNKS, BM25W_TEXTS = 200, 100   # word-BM25 candidates read per query\nRRF_K = 60\nINPUT = os.environ.get(\'INPUT_ROOT\', \'/kaggle/input\')\nOUT = Path(os.environ.get(\'OUT_DIR\', \'/kaggle/working\'))\n\n\ndef one_match(pattern):\n    hits = sorted(p for p in glob.glob(f\'{INPUT}/**/{pattern}\', recursive=True))\n    if len(hits) != 1:\n        for root in sorted(glob.glob(f\'{INPUT}/*/*/*\')) or sorted(glob.glob(f\'{INPUT}/*\')):\n            print(\'input:\', root, sorted(os.listdir(root))[:8] if os.path.isdir(root) else \'\')\n    assert len(hits) == 1, f\'{pattern}: expected one match, found {hits[:10]}\'\n    return Path(hits[0])\n\n\ndef load_submission():\n    # An uploaded dataset is unzipped by Kaggle: accept the JSON too.\n    hits = sorted(glob.glob(f\'{INPUT}/**/medxir-hybrid-submission.json\', recursive=True))\n    if hits:\n        return json.loads(Path(hits[0]).read_text(encoding=\'utf-8\'))\n    with zipfile.ZipFile(one_match(\'medxir-hybrid-submission.zip\')) as archive:\n        return json.loads(archive.read(archive.namelist()[0]))\n\n\ndef merge_run(rows):\n    """One text out of chunks of one document text with consecutive orders (rows sorted by order)."""\n    text, end = rows[0][\'text\'], rows[0][\'end\']\n    for row in rows[1:]:\n        text += row[\'text\'][max(0, end - row[\'start\']):]\n        end = row[\'end\']\n    return text\n\n\ndef build_query(ranked, n_neighbour, budget, rows, by_pos, docs_of, quota=0.0, scores=None, min_score=0.0):\n    """Selected chunk ids for one query: ranked chunks (+ neighbours of the top ones) until the budget is used.\n\n    quota > 0: a first pass caps VI chunks at (1 - quota) of the budget so ZH/EN chunks further down the pool\n    get the rest; a second pass fills what ZH/EN could not use.\n    """\n    chosen, used, vi_used = [], 0, 0\n    for vi_cap in ([budget * (1 - quota)] if quota else []) + [budget]:\n        for rank, cid in enumerate(ranked):\n            if min_score and scores[rank] < min_score:\n                break   # pool is in score order\n            group = [cid]\n            if rank < n_neighbour:\n                sha, order = rows[cid][\'sha\'], rows[cid][\'order\']\n                group += [by_pos[(sha, o)] for o in (order - 1, order + 1) if (sha, o) in by_pos]\n            for c in group:\n                if c in chosen:\n                    continue\n                cost = len(rows[c][\'text\'].encode(\'utf-8\')) * len(docs_of(c))\n                vi = rows[c].get(\'lang\', \'vi\') == \'vi\'\n                if used + cost > budget or (vi and vi_used + cost > vi_cap):\n                    continue  # a long or wide-fanout chunk does not fit; smaller ones further down may\n                chosen.append(c)\n                used += cost\n                vi_used += cost if vi else 0\n    return chosen\n\n\ndef doc_list(docs, ranked, docs_of, lang_of, n_docs=0, doc_quota=0.0):\n    """Submitted documents with the ZH/EN quota applied, then extended to n_docs with pool documents."""\n    docs = list(docs)\n    pool_docs = list(dict.fromkeys(d for cid in ranked for d in docs_of(cid)))\n    if doc_quota:\n        listed = set(docs)\n        extra = [d for d in pool_docs if d not in listed and lang_of.get(d, \'vi\') != \'vi\']\n        need = math.ceil(doc_quota * len(docs)) - sum(lang_of.get(d, \'vi\') != \'vi\' for d in docs)\n        vi_slots = [i for i, d in enumerate(docs) if lang_of.get(d, \'vi\') == \'vi\']\n        slots = vi_slots[len(vi_slots) - min(max(need, 0), len(extra), len(vi_slots)):]\n        for i, doc in zip(slots, extra):   # lowest VI slots, best ZH/EN document first\n            docs[i] = doc\n    if n_docs > len(docs):\n        listed = set(docs)\n        docs += [d for d in pool_docs if d not in listed][:n_docs - len(docs)]\n    return docs\n\n\ndef best_chunks(ranked, docs_of):\n    """{doc: its pool chunks, best first}. Documents filled from beyond the reranked pool have none."""\n    best = {}\n    for cid in ranked:\n        for doc in docs_of(cid):\n            best.setdefault(doc, []).append(cid)\n    return best\n\n\ndef doc_text(best, rows, by_text, lead_chars=400, n_chunks=1):\n    """Text a document reranker sees: the start of the document text (title / lead) + its best pool chunks.\n\n    best: the document\'s pool chunks, best first; the n_chunks best are shown in text order.\n    """\n    shown = sorted(best[:n_chunks], key=lambda c: rows[c][\'order\'])\n    parts = [rows[c][\'text\'] for c in shown]\n    lead = rows[by_text[rows[shown[0]][\'sha\']][0]]\n    if lead[\'order\'] != rows[shown[0]][\'order\']:\n        parts.insert(0, lead[\'text\'][:lead_chars])\n    return \'\\n...\\n\'.join(parts)\n\n\ndef fuse_docs(candidates, scores, weights, w_base=1.0, k=RRF_K, offset=0):\n    """Candidates (submitted order) re-ordered by weighted RRF of their position and reranker scores.\n\n    scores: {column: {doc: score}}; a document without a score in a column gets no term from it.\n    offset: submitted rank of the first candidate (when the documents above it are kept fixed).\n    """\n    fused = {doc: w_base / (k + offset + rank) for rank, doc in enumerate(candidates)}\n    for column, weight in weights.items():\n        scored = sorted((d for d in candidates if d in scores[column]), key=lambda d: -scores[column][d])\n        for rank, doc in enumerate(scored):\n            fused[doc] += weight / (k + rank)\n    return sorted(candidates, key=lambda d: -fused[d])\n\n\ndef spans(chosen, rows):\n    """Merge chosen chunks of the same text with consecutive orders. Returns [(first chunk id, text)] in rank order."""\n    by_sha = {}\n    for cid in chosen:\n        by_sha.setdefault(rows[cid][\'sha\'], []).append(cid)\n    first_rank = {cid: i for i, cid in enumerate(chosen)}\n    out = []\n    for cids in by_sha.values():\n        cids = sorted(cids, key=lambda c: rows[c][\'order\'])\n        run = [cids[0]]\n        for c in cids[1:]:\n            if rows[c][\'order\'] == rows[run[-1]][\'order\'] + 1:\n                run.append(c)\n            else:\n                out.append(run)\n                run = [c]\n        out.append(run)\n    out.sort(key=lambda run: min(first_rank[c] for c in run))\n    return [(run[0], merge_run([rows[c] for c in run])) for run in out]\n\n\ndef build_whole_docs(ranked, docs, budget, rows, by_text, docs_of):\n    """Probe: every chunk of the submitted documents, best document first, until the budget is used.\n\n    Tells whether chunk recall is lost inside the right documents (recall jumps) or by missing documents.\n    """\n    text_of = {}\n    for cid in ranked:\n        for doc in docs_of(cid):\n            text_of.setdefault(doc, rows[cid][\'sha\'])\n    chosen, used = [], 0\n    for doc in docs:\n        if doc not in text_of:\n            continue\n        for c in by_text[text_of[doc]]:\n            cost = len(rows[c][\'text\'].encode(\'utf-8\')) * len(docs_of(c))\n            if used + cost > budget:\n                return chosen\n            if c not in chosen:\n                chosen.append(c)\n                used += cost\n    return chosen\n\n\ndef load_inputs(pool_path=None, base=None, chunk_files=None):\n    """Pool ranking, base submission, every chunk of the candidate texts and the document groups.\n\n    The hybrid notebook passes its own pool.parquet, in-memory submission and validated chunk parts;\n    otherwise they are found under INPUT.\n    """\n    pool_path = pool_path or one_match(\'pool.parquet\')\n    columns = [\'query_id\', \'chunk_id\', \'final_rank\', \'final_score\']\n    pool = pd.read_parquet(pool_path, columns=columns + ([\'rrf\'] if \'rrf\' in pq.read_schema(pool_path).names else []))\n    pool = pool.sort_values([\'query_id\', \'final_rank\'])\n    ranked = {qid: g[\'chunk_id\'].tolist() for qid, g in pool.groupby(\'query_id\', sort=False)}\n    score = {qid: g[\'final_score\'].tolist() for qid, g in pool.groupby(\'query_id\', sort=False)}\n    rrf = {qid: g[\'rrf\'].tolist() for qid, g in pool.groupby(\'query_id\', sort=False)} if \'rrf\' in pool else None\n    base = load_submission() if base is None else base\n    needed_sha_prefix = {cid.rsplit(\'-\', 1)[0] for ids in ranked.values() for cid in ids}\n    bm25w = load_bm25w()\n    if bm25w:\n        needed_sha_prefix |= {cid.rsplit(\'-\', 1)[0] for ids, _ in bm25w[\'chunks\'].values() for cid in ids}\n        needed_sha_prefix |= {sha[:16] for shas, _ in bm25w[\'texts\'].values() for sha in shas}\n\n    # Every chunk of the candidate texts: the neighbours are rows of the same text_sha256.\n    rows, by_pos = {}, {}\n    chunk_files = chunk_files or sorted(glob.glob(f\'{INPUT}/**/chunks/*/part-*.parquet\', recursive=True))\n    assert chunk_files, \'no chunk parts: attach medxir-chunk-corpus\'\n    columns = [\'chunk_id\', \'text_sha256\', \'rep_doc_id\', \'order\', \'start\', \'end\', \'chunk_text\']\n    for path in chunk_files:\n        table = pq.read_table(path, columns=columns)\n        prefix = pc.utf8_slice_codeunits(table[\'text_sha256\'], 0, 16)\n        table = table.filter(pc.is_in(prefix, value_set=pa.array(sorted(needed_sha_prefix))))\n        lang = Path(path).parent.name   # chunks/<lang>/part-*.parquet\n        for cid, sha, doc, order, start, end, text in zip(*(table[c].to_pylist() for c in columns)):\n            rows[cid] = dict(sha=sha, doc=int(doc), order=int(order), start=int(start), end=int(end), text=text,\n                             lang=lang)\n            by_pos[(sha, int(order))] = cid\n        print(Path(path).parent.name, Path(path).name, f\'{len(rows):,} rows kept\', flush=True)\n    missing = {cid for ids in ranked.values() for cid in ids} - set(rows)\n    assert not missing, f\'{len(missing)} pool chunks not in the chunk corpus (chunk id != sha prefix?), e.g. {sorted(missing)[:3]}\'\n\n    groups = pd.read_parquet(one_match(\'hash_groups.parquet\'), columns=[\'text_sha256\', \'doc_ids\', \'fanout_safe\'])\n    group = {sha: [int(d) for d in ids] for sha, ids, safe in\n             zip(groups[\'text_sha256\'], groups[\'doc_ids\'], groups[\'fanout_safe\']) if bool(safe)}\n    del groups\n\n    def docs_of(cid):\n        return group.get(rows[cid][\'sha\']) or [rows[cid][\'doc\']]\n\n    by_text = {}\n    for cid in sorted(rows, key=lambda c: rows[c][\'order\']):\n        by_text.setdefault(rows[cid][\'sha\'], []).append(cid)\n\n    def text_docs(sha):\n        return group.get(sha) or ([rows[by_text[sha][0]][\'doc\']] if sha in by_text else [])\n\n    if bm25w:   # word-BM25 chunks whose rows were not found (other corpus parts) are dropped\n        for q, (ids, scores) in bm25w[\'chunks\'].items():\n            keep = [(c, s) for c, s in zip(ids, scores) if c in rows]\n            bm25w[\'chunks\'][q] = ([c for c, _ in keep], [s for _, s in keep])\n    return dict(ranked=ranked, score=score, rrf=rrf, bm25w=bm25w, text_docs=text_docs,\n                base=base, rows=rows, by_pos=by_pos, by_text=by_text, docs_of=docs_of,\n                lang_of={doc: row[\'lang\'] for cid, row in rows.items() for doc in docs_of(cid)})\n\n\ndef load_bm25w(n_chunks=None, n_texts=None):\n    """Word-BM25 candidates of bm25_words.py, {\'chunks\': {qid: (ids, scores)}, \'texts\': {qid: (shas, scores)}}, or\n    None when bm25w-chunks-k500.npz is not attached. Only positive scores, the BM25W_CHUNKS / BM25W_TEXTS best."""\n    if not glob.glob(f\'{INPUT}/**/bm25w-chunks-k500.npz\', recursive=True):\n        return None\n    out = {}\n    for key, name, column, n in ((\'chunks\', \'bm25w-chunks-k500.npz\', \'chunk_ids\', n_chunks or BM25W_CHUNKS),\n                                 (\'texts\', \'bm25w-texts-k500.npz\', \'text_sha256\', n_texts or BM25W_TEXTS)):\n        with np.load(one_match(name), allow_pickle=True) as z:\n            out[key] = {int(q): ([str(x) for x, s in zip(ids[:n], sc[:n]) if s > 0], [float(s) for s in sc[:n] if s > 0])\n                        for q, ids, sc in zip(z[\'query_ids\'], z[column], z[\'scores\'])}\n    return out\n\n\ndef load_doc_scores():\n    """{query_id: {column: {doc: score}}} from doc_rank.parquet (OUT_DIR first, then the inputs)."""\n    path = OUT / \'doc_rank.parquet\'\n    frame = pd.read_parquet(path if path.exists() else one_match(\'doc_rank.parquet\'))\n    columns = [c for c in frame.columns if c not in (\'query_id\', \'doc_id\', \'base_rank\')]\n    out = {}\n    for qid, g in frame.groupby(\'query_id\'):\n        out[int(qid)] = {c: {int(d): float(s) for d, s in zip(g[\'doc_id\'], g[c]) if s == s} for c in columns}\n    return out\n\n\ndef confidence_budget(budget, score, gamma, threshold):\n    """Per-query budgets re-weighted by (confidence / median) ** -gamma, the total kept."""\n    conf = {qid: sum(s >= threshold for s in score.get(qid, [])) + 1 for qid in budget}\n    median = sorted(conf.values())[len(conf) // 2]\n    weight = {qid: min(max((c / median) ** -gamma, CONF_CLIP[0]), CONF_CLIP[1]) for qid, c in conf.items()}\n    norm = sum(budget.values()) / sum(budget[q] * weight[q] for q in budget)\n    return {qid: budget[qid] * weight[qid] * norm for qid in budget}\n\n\ndef cover_order(ranked, n_head, docs, first, last, docs_of):\n    """Pool order with the best chunk of each listed document at ranks first..last moved right after the top n_head\n    chunks, unless a top chunk already belongs to it."""\n    head = ranked[:n_head]\n    seen = {d for c in head for d in docs_of(c)}\n    best = {}\n    for cid in ranked[n_head:]:\n        for doc in docs_of(cid):\n            best.setdefault(doc, cid)\n    promoted = list(dict.fromkeys(best[d] for d in docs[first:last] if d in best and d not in seen))\n    moved = set(promoted)\n    return head + promoted + [c for c in ranked[n_head:] if c not in moved]\n\n\ndef blended_order(ranked, columns, weights):\n    """Chunks sorted by sum(weight * column), each column min-max scaled within the query."""\n    def scaled(values):\n        lo, hi = min(values), max(values)\n        return [(v - lo) / (hi - lo) if hi > lo else 0.0 for v in values]\n    blend = [sum(w * v for w, v in zip(weights, row)) for row in zip(*(scaled(c) for c in columns))]\n    return [ranked[i] for i in sorted(range(len(ranked)), key=lambda i: -blend[i])]\n\n\ndef with_bm25w(ranked, bge, rrf, bm25w):\n    """Pool chunks + the word-BM25 chunks outside the pool, with (bge, rrf, bm25w) columns; absent scores are the\n    column minimum (0 for bge and BM25, the pool minimum for rrf)."""\n    ids, scores = bm25w\n    by_id = dict(zip(ids, scores))\n    extra = [c for c in ids if c not in set(ranked)]\n    out = list(ranked) + extra\n    rrf_floor = min(rrf) if rrf else 0.0\n    return (out, list(bge) + [0.0] * len(extra), list(rrf) + [rrf_floor] * len(extra),\n            [by_id.get(c, 0.0) for c in out])\n\n\ndef window_ids(cid, context, rows, by_pos):\n    """Chunk ids of the window order - context .. order + context of the chunk\'s text, text order."""\n    sha, order = rows[cid][\'sha\'], rows[cid][\'order\']\n    return [by_pos[(sha, o)] for o in range(order - context, order + context + 1) if (sha, o) in by_pos]\n\n\ndef build_windows(order, docs, n_cover, context, budget, rows, by_pos, docs_of):\n    """[(centre, window chunk ids)] for one query until the budget is used.\n\n    Passes: the best chunk of each of the top n_cover documents, then the chunks in order with one window per\n    text, then the chunks in order again (second windows). A centre inside a chosen window is skipped.\n    """\n    best = {}\n    for cid in order:\n        for doc in docs_of(cid):\n            best.setdefault(doc, cid)\n    head = list(dict.fromkeys(best[d] for d in docs[:n_cover] if d in best))\n    chosen, covered, used = [], {}, 0\n    for one_per_text, candidates in ((True, head), (True, order), (False, order)):\n        for cid in candidates:\n            sha, o = rows[cid][\'sha\'], rows[cid][\'order\']\n            if o in covered.get(sha, ()) or (one_per_text and sha in covered):\n                continue\n            ids = window_ids(cid, context, rows, by_pos)\n            cost = len(merge_run([rows[c] for c in ids]).encode(\'utf-8\')) * len(docs_of(cid))\n            if used + cost > budget:\n                continue   # a long or wide-fanout window does not fit; smaller ones further down may\n            chosen.append((cid, ids))\n            used += cost\n            covered.setdefault(sha, set()).update(range(o - context, o + context + 1))\n    return chosen\n\n\ndef conf_doc_count(q_scores, column, threshold, lo, hi):\n    """Documents with reranker score >= threshold, clipped to [lo, hi]."""\n    return min(max(sum(s >= threshold for s in q_scores.get(column, {}).values()), lo), hi)\n\n\ndef chosen_bytes(chosen, rows, docs_of):\n    return sum(len(rows[c][\'text\'].encode(\'utf-8\')) * len(docs_of(c)) for c in chosen)\n\n\ndef score_cut_budget(budget, ctx, n_neighbour, min_score):\n    """Budgets after the score cut: what cut queries leave unused goes to the queries the cut did not stop,\n    in proportion to their budgets."""\n    used, full = {}, []\n    for qid, b in budget.items():\n        args = (ctx[\'ranked\'][qid], n_neighbour, b, ctx[\'rows\'], ctx[\'by_pos\'], ctx[\'docs_of\'])\n        cut = chosen_bytes(build_query(*args, scores=ctx[\'score\'][qid], min_score=min_score), ctx[\'rows\'], ctx[\'docs_of\'])\n        if cut < chosen_bytes(build_query(*args), ctx[\'rows\'], ctx[\'docs_of\']):\n            used[qid] = cut\n        else:\n            full.append(qid)\n    freed = sum(budget[q] - used[q] for q in used)\n    grow = 1 + freed / max(sum(budget[q] for q in full), 1)\n    print(f\'score cut {min_score}: {len(used)} queries cut, {freed / 1e6:.1f} MB moved to {len(full)} queries\', flush=True)\n    return {qid: used[qid] if qid in used else budget[qid] * grow for qid in budget}\n\n\ndef build_variants(ctx, only=None, doc_scores=None):\n    ranked, base, rows, by_pos, by_text, docs_of, lang_of = (\n        ctx[k] for k in (\'ranked\', \'base\', \'rows\', \'by_pos\', \'by_text\', \'docs_of\', \'lang_of\'))\n    base_bytes = {e[\'id\']: sum(len(c[\'chunk_text\'].encode(\'utf-8\')) for c in e[\'relevant_chunks\'])\n                  for e in base}\n    only = ONLY if only is None else only\n    manifest = {\'budget_scale\': BUDGET_SCALE, \'variants\': {}}\n    for name, spec in VARIANTS.items():\n        if only and name not in only:\n            continue\n        if spec.get(\'win\') is not None and ctx.get(\'rrf\') is None:\n            print(name, \'skipped: pool.parquet has no rrf column\', flush=True)\n            continue\n        uses_bm25w = \'bm25w\' in spec.get(\'doc_rr\', {}) or spec.get(\'w_bm25w\')\n        if uses_bm25w and not ctx.get(\'bm25w\'):\n            print(name, \'skipped: bm25w-chunks-k500.npz not attached\', flush=True)\n            continue\n        if spec.get(\'doc_rr\'):\n            doc_scores = doc_scores or load_doc_scores()\n            some = next(iter(doc_scores.values()))\n            absent = [c for c in spec[\'doc_rr\'] if c not in some and c != \'bm25w\']\n            if absent:\n                print(name, \'skipped: no doc_rank column\', absent, flush=True)\n                continue\n        n_neighbour = spec[\'nb\']\n        budget = {qid: b * spec.get(\'budget_scale\', BUDGET_SCALE) for qid, b in base_bytes.items()}\n        q_budget = (confidence_budget(budget, ctx[\'score\'], spec[\'conf_gamma\'], spec.get(\'conf_t\', 0.5))\n                    if spec.get(\'conf_gamma\') else budget)\n        if spec.get(\'min_score\'):\n            q_budget = score_cut_budget(q_budget, ctx, n_neighbour, spec[\'min_score\'])\n        scale = 1.0\n        while True:\n            submission, n_spans, n_chunks, n_docs, other_docs, other_bytes, all_bytes, moved = [], 0, 0, 0, 0, 0, 0, 0\n            dropped = 0\n            for entry in base:\n                qid = entry[\'id\']\n                if spec.get(\'doc_rr\'):\n                    q_scores = doc_scores.get(qid, {})\n                    if \'bm25w\' in spec[\'doc_rr\']:\n                        q_scores = dict(q_scores, bm25w={})\n                        for sha, s in zip(*ctx[\'bm25w\'][\'texts\'].get(qid, ([], []))):\n                            for doc in ctx[\'text_docs\'](sha):\n                                q_scores[\'bm25w\'].setdefault(doc, s)\n                    candidates = list(dict.fromkeys(\n                        doc_list(entry[\'relevant_docs\'], ranked[qid], docs_of, lang_of, n_docs=10 ** 6)\n                        + [d for c in spec[\'doc_rr\'] for d in q_scores.get(c, {})]))\n                    keep = spec.get(\'n_docs\') or len(entry[\'relevant_docs\'])\n                    if spec.get(\'doc_conf\'):\n                        keep = conf_doc_count(q_scores, next(iter(spec[\'doc_rr\'])), *spec[\'doc_conf\'])\n                    fixed = candidates[:spec.get(\'keep_top\', 0)]\n                    docs = fixed + fuse_docs(candidates[len(fixed):], q_scores, spec[\'doc_rr\'], spec.get(\'w_base\', 1.0),\n                                             offset=len(fixed))[:keep - len(fixed)]\n                    moved += len(set(docs) - set(entry[\'relevant_docs\']))\n                else:\n                    docs = doc_list(entry[\'relevant_docs\'], ranked[qid], docs_of, lang_of,\n                                    spec.get(\'n_docs\', 0), spec.get(\'doc_quota\', 0.0))\n                windows = None\n                if spec.get(\'win\') is not None:\n                    if spec.get(\'w_bm25w\'):\n                        ids, bge, rrf, bm = with_bm25w(ranked[qid], ctx[\'score\'][qid], ctx[\'rrf\'][qid],\n                                                       ctx[\'bm25w\'][\'chunks\'].get(qid, ([], [])))\n                        weights = [spec[\'w_bge\'], 1 - spec[\'w_bge\'] - spec[\'w_bm25w\'], spec[\'w_bm25w\']]\n                        order = blended_order(ids, [bge, rrf, bm], weights)\n                    else:\n                        order = blended_order(ranked[qid], [ctx[\'score\'][qid], ctx[\'rrf\'][qid]],\n                                              [spec[\'w_bge\'], 1 - spec[\'w_bge\']])\n                    windows = build_windows(order, docs, spec.get(\'cover_top\', 0), spec[\'win\'], q_budget[qid] * scale,\n                                            rows, by_pos, docs_of)\n                    chosen = list(dict.fromkeys(c for _, ids in windows for c in ids))\n                elif n_neighbour == \'whole\':\n                    chosen = build_whole_docs(ranked[qid], entry[\'relevant_docs\'], q_budget[qid] * scale,\n                                              rows, by_text, docs_of)\n                elif spec.get(\'cover\'):\n                    order = cover_order(ranked[qid], n_neighbour, docs, *spec[\'cover\'], docs_of)\n                    chosen = build_query(order, n_neighbour, q_budget[qid] * scale, rows, by_pos, docs_of)\n                else:\n                    chosen = build_query(ranked[qid], n_neighbour, q_budget[qid] * scale, rows, by_pos, docs_of,\n                                         spec.get(\'quota\', 0.0), ctx[\'score\'][qid], spec.get(\'min_score\', 0.0))\n                for c in chosen:\n                    size = len(rows[c][\'text\'].encode(\'utf-8\'))\n                    all_bytes += size\n                    other_bytes += size if rows[c][\'lang\'] != \'vi\' else 0\n                listed = set(docs)\n                chunks = []\n                entries = ([(cid, merge_run([rows[c] for c in ids])) for cid, ids in windows] if windows is not None\n                           else spans(chosen, rows))\n                for cid, text in entries:\n                    if len(text) < spec.get(\'min_chars\', 0):\n                        dropped += 1\n                        continue\n                    for doc in docs_of(cid):\n                        chunks.append(dict(doc_id=doc, chunk_text=text, chunk_order=rows[cid][\'order\']))\n                        if doc not in listed:\n                            listed.add(doc)\n                            docs.append(doc)\n                n_spans += len(chunks)\n                n_docs += len(docs)\n                other_docs += sum(lang_of.get(d, \'vi\') != \'vi\' for d in docs) / max(len(docs), 1)\n                n_chunks += len(chosen)\n                submission.append(dict(id=qid, relevant_docs=docs, relevant_chunks=chunks))\n            zip_path = OUT / f\'medxir-expand-{name}-submission.zip\'\n            json_name = zip_path.name.replace(\'.zip\', \'.json\')\n            method = zipfile.ZIP_BZIP2 if spec.get(\'zip\') == \'bzip2\' else zipfile.ZIP_DEFLATED\n            with zipfile.ZipFile(zip_path, \'w\', compression=method, compresslevel=9) as archive:\n                archive.writestr(json_name, json.dumps(submission, ensure_ascii=False, separators=(\',\', \':\')))\n            mb = zip_path.stat().st_size / 1e6\n            if mb <= MAX_ZIP_MB:\n                break\n            scale *= 0.95 * MAX_ZIP_MB / mb\n        stats = dict(spec, zip_mb=round(mb, 1), budget_scale_applied=round(scale, 3),\n                     chunks_per_query=round(n_chunks / len(base), 1), entries_per_query=round(n_spans / len(base), 1),\n                     docs_per_query=round(n_docs / len(base), 1), short_dropped_per_query=round(dropped / len(base), 1),\n                     zh_en_doc_share=round(other_docs / len(base), 3),\n                     zh_en_chunk_byte_share=round(other_bytes / max(all_bytes, 1), 3))\n        if spec.get(\'doc_rr\'):\n            stats[\'new_docs_per_query\'] = round(moved / len(base), 1)   # documents not in the submitted list\n        manifest[\'variants\'][name] = stats\n        print(name, stats, flush=True)\n    (OUT / \'expand_manifest.json\').write_text(json.dumps(manifest, indent=2))\n    return manifest\n\n\ndef main():\n    build_variants(load_inputs())\n\n\nif __name__ == \'__main__\':\n    main()\n', 'reranker_inputs.py': '"""Qwen reranker inputs: preserve the classification suffix when truncating.\n\nMatches the official Qwen3-Reranker Transformers recipe: reserve prompt tokens,\ntruncate the body, append the untouched prefix/suffix, and left-pad the batch.\n"""\n\nQWEN_RERANK_PREFIX = (\'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query and the \'\n                     \'Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\\n<|im_start|>user\\n\')\nQWEN_RERANK_SUFFIX = \'<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\'\n\n\ndef qwen_reranker_inputs(tokenizer, queries, documents, instruction, max_length, return_tensors=\'pt\'):\n    if len(queries) != len(documents) or not queries:\n        raise ValueError(\'Expected equal nonempty query/document batches\')\n    if tokenizer.padding_side != \'left\':\n        raise ValueError(\'Qwen last-position scoring requires left padding\')\n    prefix = tokenizer.encode(QWEN_RERANK_PREFIX, add_special_tokens=False)\n    suffix = tokenizer.encode(QWEN_RERANK_SUFFIX, add_special_tokens=False)\n    available = max_length - len(prefix) - len(suffix)\n    if available < 1:\n        raise ValueError(\'Token limit leaves no room for query/document body\')\n    bodies = [f\'<Instruct>: {instruction}\\n<Query>: {q}\\n<Document>: {d}\' for q,d in zip(queries,documents)]\n    encoded = tokenizer(bodies, padding=False, truncation=True, add_special_tokens=False,\n                        return_attention_mask=False, max_length=available)\n    ids = [prefix + body + suffix for body in encoded[\'input_ids\']]\n    assert all(len(row) <= max_length and row[-len(suffix):] == suffix for row in ids)\n    return tokenizer.pad({\'input_ids\': ids}, padding=True, return_tensors=return_tensors)\n', 'evidence_spans.py': '"""Contiguous source passages for reranking; offsets always refer to original chunk text.\n\nThese helpers do not change the frozen chunk corpus or embeddings.\n"""\nfrom bisect import bisect_left, bisect_right\nimport re\n\n\ndef join_overlapping(rows):\n    """Return (original start, source text); reject unknown gaps/inconsistent overlaps."""\n    if not rows:\n        raise ValueError(\'No source rows\')\n    rows=sorted(rows,key=lambda r:r[\'start\'])\n    start=rows[0][\'start\'];text=rows[0][\'text\'];end=rows[0][\'end\']\n    sha=rows[0][\'sha\']\n    for row in rows:\n        if row[\'sha\']!=sha or row[\'end\']-row[\'start\']!=len(row[\'text\']):\n            raise ValueError(\'Invalid source identity or offsets\')\n    for row in rows[1:]:\n        if row[\'start\']>end:\n            raise ValueError(\'Unknown source gap; cannot invent separator text\')\n        overlap=min(end,row[\'end\'])-row[\'start\']\n        if text[row[\'start\']-start:row[\'start\']-start+overlap]!=row[\'text\'][:overlap]:\n            raise ValueError(\'Source chunks disagree in overlapping region\')\n        if row[\'end\']>end:\n            text+=row[\'text\'][end-row[\'start\']:];end=row[\'end\']\n    return start,text\n\n\ndef contiguous_around_anchor(rows, anchor):\n    """Keep the maximal known contiguous component containing the seed.\n\n    A distant gap must not discard the seed\'s valid left/right neighbours.\n    Unknown characters across a gap are never reconstructed or guessed.\n    """\n    if not rows or not any(row == anchor for row in rows):\n        raise ValueError(\'Anchor must be one of the source rows\')\n    components=[]\n    for row in sorted(rows,key=lambda r:r[\'start\']):\n        if row[\'sha\'] != anchor[\'sha\']:\n            raise ValueError(\'Invalid source identity\')\n        if not components or row[\'start\'] > max(r[\'end\'] for r in components[-1]):\n            components.append([row])\n        else:\n            components[-1].append(row)\n    # Validate all components, including those outside the selected region.\n    joined=[join_overlapping(component) for component in components]\n    for start,text in joined:\n        if start <= anchor[\'start\'] and start+len(text) >= anchor[\'end\']:\n            if text[anchor[\'start\']-start:anchor[\'end\']-start] != anchor[\'text\']:\n                raise ValueError(\'Anchor disagrees with source component\')\n            return start,text\n    raise ValueError(\'Anchor not covered by source component\')\n\n\ndef centered_span(text, offsets, anchor_start, anchor_end, max_tokens):\n    """A token-budgeted, contiguous source slice around a retrieved seed\'s centre.\n\nUse tokenizer offset_mapping, not decoded tokens (which can rewrite whitespace).\nSnap inward to sentence boundaries only within 10% of the token budget.\n"""\n    if not text or not offsets or max_tokens<1 or not 0<=anchor_start<anchor_end<=len(text):\n        raise ValueError(\'Invalid source/anchor/token budget\')\n    offsets=[(int(a),int(b)) for a,b in offsets if b>a]\n    if not offsets:raise ValueError(\'No nonempty token offsets\')\n    if len(offsets)<=max_tokens:return 0,len(text),text\n    midpoint=(anchor_start+anchor_end)//2\n    centre=max(0,bisect_right([a for a,_ in offsets],midpoint)-1)\n    lo=max(0,min(centre-max_tokens//2,len(offsets)-max_tokens));hi=lo+max_tokens\n    begin=offsets[lo][0];end=offsets[hi-1][1]\n    margin=max(1,max_tokens//10)\n    left_limit=offsets[min(lo+margin,hi-1)][0]\n    right_limit=offsets[max(lo,hi-margin)][0]\n    boundaries=[m.end() for m in re.finditer(r\'(?:[.!?。！？](?:\\s+|(?=[\\u3400-\\u9fff]))|\\n+)\',text[begin:end])]\n    left=[begin+x for x in boundaries if begin+x<=left_limit and begin+x<midpoint]\n    right=[begin+x for x in boundaries if begin+x>=right_limit and begin+x>midpoint]\n    if left:begin=left[0]\n    if right:end=right[-1]\n    assert 0<=begin<end<=len(text) and begin<=midpoint<=end\n    return begin,end,text[begin:end]\n', 'pilot_evidence_rerank.py': '"""Bounded Kaggle GPU experiment; score source passages, never submit to leaderboard."""\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM\n\nimport expand_chunks as ex\nfrom evidence_spans import join_overlapping,centered_span,contiguous_around_anchor\nfrom reranker_inputs import qwen_reranker_inputs,QWEN_RERANK_PREFIX,QWEN_RERANK_SUFFIX\n\nN_QUERIES=16\nDOC_DEPTH=400\nPASSAGE_TOKENS=768\nPAIR_TOKENS=2048\nINSTRUCTION=\'Given a web search query, retrieve relevant passages that answer the query\'\n\n\ndef main():\n    started=time.monotonic();out=Path(\'/kaggle/working\')\n    gpu=torch.cuda.get_device_name(0);assert \'6000\' in gpu,gpu\n    print(\'GPU:\',gpu,flush=True)\n    pool_path=ex.one_match(\'pool.parquet\');pool=pd.read_parquet(pool_path)\n    qids=sorted(pool.query_id.unique(),key=lambda q:hashlib.sha256(str(q).encode()).hexdigest())[:N_QUERIES]\n    pool_sample=out/\'pilot_pool.parquet\';pool[pool.query_id.isin(qids)].to_parquet(pool_sample,index=False)\n    base=[e for e in ex.load_submission() if e[\'id\'] in set(qids)]\n    ctx=ex.load_inputs(pool_path=pool_sample,base=base)\n    queries=dict(pd.read_parquet(ex.one_match(\'query.parquet\'),columns=[\'id\',\'query\']).itertuples(index=False,name=None))\n    paths={key:ex.one_match(name+\'/config.json\').parent for key,name in\n           [(\'bge\',\'bge-reranker-v2-m3\'),(\'qwen\',\'Qwen3-Reranker-4B\')]}\n    tokenizer=AutoTokenizer.from_pretrained(paths[\'bge\'],local_files_only=True)\n    pairs=[];metadata=[];legacy_docs=[];skips=[];seed_only=0;component_recovered=0\n    for entry in base:\n        qid=int(entry[\'id\']);query=str(queries[qid]);ranked=ctx[\'ranked\'][qid]\n        seed_rank={c:i for i,c in enumerate(ranked)}\n        best=ex.best_chunks(ranked,ctx[\'docs_of\'])\n        candidates=ex.doc_list(entry[\'relevant_docs\'],ranked,ctx[\'docs_of\'],ctx[\'lang_of\'],n_docs=10**6)[:DOC_DEPTH]\n        done=set()\n        for doc in candidates:\n            if doc not in best:continue\n            sha=ctx[\'rows\'][best[doc][0]][\'sha\']\n            if sha in done:continue\n            done.add(sha)\n            selected=[]\n            # At most two distinct seeds; no per-query hyperparameters or labels.\n            for cid in best[doc]:\n                if all(abs(ctx[\'rows\'][cid][\'order\']-ctx[\'rows\'][c][\'order\'])>1 for c in selected):\n                    selected.append(cid)\n                if len(selected)==2:break\n            used_spans=set()\n            for cid in selected:\n                row=ctx[\'rows\'][cid]\n                ids=ex.window_ids(cid,2,ctx[\'rows\'],ctx[\'by_pos\'])\n                try:source_start,source=join_overlapping([ctx[\'rows\'][c] for c in ids])\n                except ValueError as err:\n                    if \'Unknown source gap\' not in str(err):raise\n                    skips.append(dict(qid=qid,chunk_id=cid,reason=str(err)))\n                    source_start,source=contiguous_around_anchor([ctx[\'rows\'][c] for c in ids],row)\n                    component_recovered+=int(len(source)>len(row[\'text\']))\n                seed_only+=int(source_start==row[\'start\'] and source==row[\'text\'])\n                offsets=tokenizer(source,add_special_tokens=False,return_offsets_mapping=True)[\'offset_mapping\']\n                a,b,text=centered_span(source,offsets,row[\'start\']-source_start,row[\'end\']-source_start,PASSAGE_TOKENS)\n                # Re-tokenize exact final slice, rather than assume token counts are additive.\n                tokens=len(tokenizer.encode(text,add_special_tokens=False))\n                if tokens>PASSAGE_TOKENS:\n                    a,b,text=centered_span(source,offsets,row[\'start\']-source_start,row[\'end\']-source_start,PASSAGE_TOKENS-8)\n                    tokens=len(tokenizer.encode(text,add_special_tokens=False))\n                assert tokens<=PASSAGE_TOKENS+2 and text==source[a:b]\n                key=(source_start+a,source_start+b)\n                if key in used_spans:continue\n                used_spans.add(key)\n                pairs.append((query,text))\n                metadata.append(dict(query_id=qid,doc_id=int(doc),text_sha256=sha,seed=cid,seed_rank=seed_rank[cid],\n                                     doc_ids=list(map(int,ctx[\'docs_of\'](cid))),start=key[0],end=key[1],\n                                     chunk_text=text,passage_tokens=tokens))\n            legacy_docs.append((query,ex.doc_text(best[doc],ctx[\'rows\'],ctx[\'by_text\'],400,1)))\n        print(\'prepared\',qid,\'pairs\',len(pairs),flush=True)\n    frame=pd.DataFrame(metadata)\n    assert len(frame) and frame.query_id.nunique()==N_QUERIES\n    frame.to_parquet(out/\'passage_candidates.parquet\',index=False)\n    report=dict(complete=False,gpu=gpu,n_queries=N_QUERIES,pairs=len(pairs),doc_depth=DOC_DEPTH,\n                passage_tokens=PASSAGE_TOKENS,pair_tokens=PAIR_TOKENS,skipped_source_gaps=skips,\n                pool_sha256=hashlib.sha256(pool_path.read_bytes()).hexdigest(),\n                score_quality=\'UNMEASURED: no gold labels\',timings={},truncations={},\n                seed_only_windows=seed_only,disconnected_windows_recovered=component_recovered,\n                source_gap_policy=\'maximal contiguous component around seed; never invent gap text\')\n    order=np.argsort([len(d) for q,d in pairs],kind=\'stable\')\n    for key in [\'bge\',\'qwen\']:\n        t0=time.monotonic();path=paths[key]\n        tok=AutoTokenizer.from_pretrained(path,local_files_only=True,padding_side=\'left\' if key==\'qwen\' else \'right\')\n        if key==\'qwen\':\n            model=AutoModelForCausalLM.from_pretrained(path,torch_dtype=torch.bfloat16,local_files_only=True,\n                                                     attn_implementation=\'sdpa\').cuda().eval()\n            yes,no=tok.convert_tokens_to_ids(\'yes\'),tok.convert_tokens_to_ids(\'no\')\n            old_pairs=legacy_docs[:128]\n            legacy_texts=[QWEN_RERANK_PREFIX+f\'<Instruct>: {INSTRUCTION}\\n<Query>: {q}\\n<Document>: {d}\'+QWEN_RERANK_SUFFIX for q,d in old_pairs]\n            old_ids=tok(legacy_texts,padding=False,truncation=True,max_length=512)[\'input_ids\']\n            suffix=tok.encode(QWEN_RERANK_SUFFIX,add_special_tokens=False)\n            report[\'legacy_qwen_suffix_lost\']=sum(row[-len(suffix):]!=suffix for row in old_ids)\n            report[\'legacy_qwen_checked\']=len(old_ids)\n            fixed=qwen_reranker_inputs(tok,[q for q,d in old_pairs],[d for q,d in old_pairs],INSTRUCTION,512,return_tensors=None)\n            assert all(row[-len(suffix):]==suffix for row in fixed[\'input_ids\'])\n            report[\'fixed_qwen_suffix_lost\']=0\n        else:\n            model=AutoModelForSequenceClassification.from_pretrained(path,torch_dtype=torch.bfloat16,local_files_only=True,\n                                                                    attn_implementation=\'sdpa\').cuda().eval()\n        scores=np.empty(len(pairs),dtype=np.float32);batch=16 if key==\'qwen\' else 32\n        truncated=0\n        with torch.inference_mode():\n            for start in range(0,len(order),batch):\n                ix=order[start:start+batch];qs=[pairs[i][0] for i in ix];ds=[pairs[i][1] for i in ix]\n                if key==\'qwen\':\n                    body_texts=[f\'<Instruct>: {INSTRUCTION}\\n<Query>: {q}\\n<Document>: {d}\' for q,d in zip(qs,ds)]\n                    raw_ids=tok(body_texts,add_special_tokens=False,truncation=False)[\'input_ids\']\n                    overhead=len(tok.encode(QWEN_RERANK_PREFIX,add_special_tokens=False))+len(tok.encode(QWEN_RERANK_SUFFIX,add_special_tokens=False))\n                    truncated+=sum(len(ids)+overhead>PAIR_TOKENS for ids in raw_ids)\n                    enc=qwen_reranker_inputs(tok,qs,ds,INSTRUCTION,PAIR_TOKENS).to(\'cuda\')\n                    logits=model(**enc,logits_to_keep=1).logits[:,-1,:]\n                    result=torch.softmax(torch.stack([logits[:,no],logits[:,yes]],dim=1).float(),dim=1)[:,1]\n                else:\n                    enc=tok(qs,ds,padding=True,truncation=False,return_tensors=\'pt\')\n                    assert enc[\'input_ids\'].shape[1]<=PAIR_TOKENS,\'BGE pair too long\'\n                    result=torch.sigmoid(model(**enc.to(\'cuda\')).logits.view(-1).float())\n                scores[ix]=result.cpu().numpy()\n                if start//batch%20==0:\n                    rate=(start+len(ix))/max(time.monotonic()-t0,.01)\n                    print(key,start+len(ix),\'/\',len(pairs),\'pairs/s\',round(rate,2),flush=True)\n        assert np.isfinite(scores).all()\n        frame[key]=scores\n        report[\'truncations\'][key]=truncated\n        report[\'timings\'][key]=dict(seconds=time.monotonic()-t0,pairs_per_second=len(pairs)/(time.monotonic()-t0))\n        frame.to_parquet(out/\'passage_scores.parquet\',index=False)\n        del model,tok;gc.collect();torch.cuda.empty_cache()\n    # Compact review artifact; model agreement is a diagnostic, not a gold metric.\n    review=[]\n    for qid,g in frame.groupby(\'query_id\'):\n        for key in [\'bge\',\'qwen\']:\n            for rank,(_,r) in enumerate(g.sort_values(key,ascending=False).head(5).iterrows()):\n                review.append(dict(query_id=int(qid),query=str(queries[qid]),model=key,rank=rank,\n                                   doc_id=int(r.doc_id),score=float(r[key]),text=r.chunk_text))\n    (out/\'passage_review.json\').write_text(json.dumps(review,ensure_ascii=False,indent=2),encoding=\'utf8\')\n    report.update(complete=True,elapsed_seconds=time.monotonic()-started,\n                  full_run_pair_estimate=len(pairs)*1200/N_QUERIES)\n    (out/\'passage_pilot_report.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\'utf8\')\n    print(json.dumps(report[\'timings\']),flush=True)\n\n\nif __name__==\'__main__\':main()\n'}
from pathlib import Path
import sys,subprocess
for name,source in FILES.items():Path('/kaggle/working',name).write_text(source,encoding='utf8')
subprocess.run([sys.executable,'-u','/kaggle/working/pilot_evidence_rerank.py'],check=True)
